# Collaborative filtering / matrix factorization

_Classical ML (beyond the cheat sheet)_

**Fill the blanks in a ratings table by factoring it into two thin matrices.**

You have a big grid: rows are users, columns are items, cells are ratings. Most cells are blank.

     Collaborative filtering fills the blanks by learning hidden tastes.

---

This notebook is a practice scaffold for the **Collaborative filtering / matrix factorization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

rng = np.random.default_rng(0)
# true low-rank ratings: 40 users x 25 items, rank 3
U = rng.normal(size=(40, 3)); V = rng.normal(size=(25, 3))
R = U @ V.T

mask = rng.random(R.shape) < 0.7                   # 70% observed
Robs = np.where(mask, R, 0.0)                       # zero-fill unknowns

svd = TruncatedSVD(n_components=3, random_state=0)
Z = svd.fit_transform(Robs)                         # user factors
Rhat = Z @ svd.components_                           # reconstruction

known = mask
hidden = ~mask
print("rank used        :", svd.n_components)
print("RMSE on observed :", round(np.sqrt(np.mean((Rhat[known]-R[known])**2)), 3))
print("RMSE on hidden   :", round(np.sqrt(np.mean((Rhat[hidden]-R[hidden])**2)), 3))

## Visualize the data & results

_Can we fill in the movie ratings these viewers never gave?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

users = ["Alice","Ben","Chloe","Diego","Emma","Frank"]
movies = ["Matrix","Titanic","Inception","ToyStory","Gladiator","Frozen","Avatar","Shrek"]
R = np.array([                          # real-style 1-5 ratings
 [5,2,5,3,4,2,4,3],[4,2,4,4,3,2,3,3],[2,5,2,4,2,5,3,5],
 [3,3,3,4,3,3,4,4],[1,4,1,3,2,5,2,5],[4,3,5,3,5,2,4,2]], dtype=float)
mask = np.array([                        # which cells the user actually rated
 [1,1,0,1,1,1,1,0],[1,0,1,1,0,1,1,1],[1,1,1,0,1,1,0,1],
 [0,1,1,1,1,0,1,1],[1,0,1,1,1,1,1,0],[1,1,0,1,1,1,0,1]], dtype=bool)

mean = R[mask].mean()
Robs = np.where(mask, R - mean, 0.0)     # center, zero-fill unknowns
svd = TruncatedSVD(n_components=3, random_state=0)
Rhat = svd.fit_transform(Robs) @ svd.components_ + mean

obs = np.where(mask, R, np.nan)          # blanks as NaN for display
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(obs, cmap="coolwarm", vmin=1, vmax=5)
ax[0].set_title("Observed movie ratings (blanks hidden)")
ax[1].imshow(Rhat, cmap="coolwarm", vmin=1, vmax=5)
ax[1].set_title("Reconstructed ratings (rank-3 SVD)")
for a in ax:
    a.set_xticks(range(len(movies))); a.set_xticklabels(movies, rotation=90)
    a.set_yticks(range(len(users))); a.set_yticklabels(users)
plt.show()